Test test 

In [26]:
import pandas as pd
import altair as alt
from sklearn import set_config
set_config(transform_output="pandas")
from sklearn.preprocessing import StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.compose import make_column_selector
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [27]:
players_url = "https://drive.google.com/uc?export=download&id=1Mw9vW0hjTJwRWx0bDXiSpYsO3gKogaPz"
players_messy=pd.read_csv(players_url)
players_messy

,experience,subscribe,hashedEmail,played_hours,name,gender,age,individualId,organizationName
0,Pro,True,f6daba428a5e19a3d47574858c13550499be23603422e6...,30.3,Morgan,Male,9,NaN,NaN
1,Veteran,True,f3c813577c458ba0dfef80996f8f32c93b6e8af1fa9397...,3.8,Christian,Male,17,NaN,NaN
2,Veteran,False,b674dd7ee0d24096d1c019615ce4d12b20fcbff12d79d3...,0.0,Blake,Male,17,NaN,NaN
3,Amateur,True,23fe711e0e3b77f1da7aa221ab1192afe21648d47d2b4f...,0.7,Flora,Female,21,NaN,NaN
4,Regular,True,7dc01f10bf20671ecfccdac23812b1b415acd42c2147cb...,0.1,Kylie,Male,21,NaN,NaN
...,...,...,...,...,...,...,...,...,...
191,Amateur,True,b6e9e593b9ec51c5e335457341c324c34a2239531e1890...,0.0,Bailey,Female,17,NaN,NaN
192,Veteran,False,71453e425f07d10da4fa2b349c83e73ccdf0fb3312f778...,0.3,Pascal,Male,22,NaN,NaN
193,Amateur,False,d572f391d452b76ea2d7e5e53a3d38bfd7499c7399db29...,0.0,Dylan,Prefer not to say,17,NaN,NaN
194,Amateur,False,f19e136ddde68f365afc860c725ccff54307dedd13968e...,2.3,Harlow,Male,17,NaN,NaN


This table consists of 9 variables, where the last two are empty:

-The player's experience: whether they are just starting to play or are already a pro at the game, 
-whether they are subscribed to a newsletter related to the game or not, 
-their email address, which is transformed for privacy, 
-the number of hours they have played, 
-their name,
-their self-reported gender, 
-their self-reported age, 
-and their individual ID and organization name, which are both empty for all observations. 
The dataset includes information from 196 players, containing both quantitative and qualitative data. Two columns, individual and organizationName, are empty and should be removed. Data was collected through in-game voice chats, and information was provided during sign-up. However, the collection method has limitations: much of the data is self-reported, making it subjective and potentially unreliable. Furthermore, requiring players to communicate vocally with strangers may deter those less comfortable with online interaction (e.g., introverted players), introducing bias toward more extroverted participants and reducing the dataset’s representativeness.

In [28]:
players_wrangled=players_messy.drop(columns=["hashedEmail", "name", "individualId", "organizationName", "gender", "played_hours"])
players_wrangled

,experience,subscribe,age
0,Pro,True,9
1,Veteran,True,17
2,Veteran,False,17
3,Amateur,True,21
4,Regular,True,21
...,...,...,...
191,Amateur,True,17
192,Veteran,False,22
193,Amateur,False,17
194,Amateur,False,17


In [29]:
experience_map = {
    "Amateur": 1,
    "Beginner": 2,
    "Regular": 3,
    "Pro": 4,
    "Veteran": 5
}

players_wrangled["experience"] = players_wrangled["experience"].map(experience_map)
players_wrangled

,experience,subscribe,age
0,4,True,9
1,5,True,17
2,5,False,17
3,1,True,21
4,3,True,21
...,...,...,...
191,1,True,17
192,5,False,22
193,1,False,17
194,1,False,17


In [30]:
experience_plot = alt.Chart(players_wrangled).mark_circle(size=60).encode(
    x=alt.X("experience:O", title="Experience level"),
    y=alt.Y("age:Q", title="Age"),
    color=alt.Color("subscribe:N", title="Subscribed"),
    tooltip=["experience", "age", "subscribe"]
).properties(
    title="Age vs Experience by Subscription Status"
)

experience_plot

alt.Chart(...)

In [31]:
age_plot=alt.Chart(players_wrangled).mark_point().encode(
    x=alt.X("age").title("player_age"),
    y=alt.Y("experience").title("experience_level"), 
    color=alt.Color("subscribe").title("subscribed or not")
)
age_plot

alt.Chart(...)

In [32]:
players_filtered = players_wrangled[(players_wrangled['age'] > 5) & (players_wrangled['age'] < 50)]
players_filtered

,experience,subscribe,age
0,4,True,9
1,5,True,17
2,5,False,17
3,1,True,21
4,3,True,21
...,...,...,...
190,1,True,20
191,1,True,17
192,5,False,22
193,1,False,17


In [33]:
filtered_experience_plot = alt.Chart(players_filtered).mark_circle(size=60).encode(
    x=alt.X("experience:O", title="Experience level"),
    y=alt.Y("age:Q", title="Age"),
    color=alt.Color("subscribe:N", title="Subscribed"),
    tooltip=["experience", "age", "subscribe"]
).properties(
    title="Age vs Experience by Subscription Status"
)

filtered_experience_plot

alt.Chart(...)

In [34]:
filtered_age_plot=alt.Chart(players_filtered).mark_circle().encode(
    x=alt.X("age").title("player age"),
    y=alt.Y("experience").title("experience level"), 
    color=alt.Color("subscribe").title("subscribed or not")
)
filtered_age_plot

alt.Chart(...)

In [35]:
preprocessor = make_column_transformer(
    (StandardScaler(), make_column_selector(dtype_include="number")),
    remainder="passthrough",
    verbose_feature_names_out=False
)
preprocessor

ColumnTransformer(remainder='passthrough',
                  transformers=[('standardscaler', StandardScaler(),
                                 <sklearn.compose._column_transformer.make_column_selector object at 0x7fcdf52e1550>)],
                  verbose_feature_names_out=False)

In [36]:
preprocessor.fit(players_filtered)
scaled_players = preprocessor.transform(players_filtered)
scaled_players

,experience,age,subscribe
0,0.806637,-1.961133,True
1,1.442070,-0.581010,True
2,1.442070,-0.581010,False
3,-1.099660,0.109051,True
4,0.171205,0.109051,True
...,...,...,...
190,-1.099660,-0.063464,True
191,-1.099660,-0.581010,True
192,1.442070,0.281566,False
193,-1.099660,-0.581010,False


In [37]:
scaled_age_plot=alt.Chart(scaled_players).mark_circle(opacity=0.5).encode(
    x=alt.X("age").title("player age"),
    y=alt.Y("experience").title("experience level"), 
    color=alt.Color("subscribe").title("subscribed or not")
)
scaled_age_plot

alt.Chart(...)

In [38]:
#split data, use a random state 
players_train, players_test = train_test_split(players_wrangled, test_size=0.25, random_state=123, stratify=players_wrangled["subscribe"])

X_train = players_train[["experience", "age"]]
y_train = players_train["subscribe"]

X_test = players_test[["experience", "age"]]
y_test = players_test["subscribe"]

In [39]:
#standardize training data 
players_preprocessor = make_column_transformer(
    (StandardScaler(), ["age","experience"]),
     verbose_feature_names_out=False
)

In [51]:
#find best k using CV 
param_grid = {
    "kneighborsclassifier__n_neighbors": range(2, 15, 1),
}

players_pipe= make_pipeline(players_preprocessor, KNeighborsClassifier())

knn_tune= GridSearchCV(
    estimator=players_pipe, 
    param_grid=param_grid, 
    cv=5,
    n_jobs=-1,
    return_train_score=True)

knn_model= knn_tune.fit(X_train, y_train)

accuracies= pd.DataFrame(knn_model.cv_results_)

In [53]:
#plot to visualize best k 
cross_val_plot = alt.Chart(accuracies).mark_line(point=True).encode(
    x=alt.X("param_kneighborsclassifier__n_neighbors")
        .title("Neighbors")
        .scale(zero=False),
    y=alt.Y("mean_test_score")
        .title("Mean Test Score")
        .scale(zero=False)
)
cross_val_plot

alt.Chart(...)

In [54]:
#confirm best k
knn_tune.best_params_

{'kneighborsclassifier__n_neighbors': 11}

In [ ]:
#run the model on the test data 
knn = KNeighborsClassifier(n_neighbors=11)

knn_pipeline = make_pipeline(players_preprocessor, knn)
knn_pipeline.fit(X_train,y_train)

players_test["predicted"] = knn_pipeline.predict(players_test[["experience", "age"]])

In [ ]:
#run the model on the test data 

#players_test["predicted"] = knn_tune_grid.predict(
#    players_test[["experience_enc", "age"]]
#)

#knn_tune_grid.score(
#   players_test[["experience_enc", "age"]],
#   players_test["subscribe"]
#)

#precision_score(
#    y_true=players_test["subscribe"],
#    y_pred=players_test["predicted"],
#    pos_label=True
#)

#recall_score(
#    y_true=players_test["subscribe"],
#    y_pred=players_test["predicted"],
#    pos_label=True
#)

#pd.crosstab(
#    players_test["subscribe"],
#    players_test["predicted"]
#)

In [ ]:
#evaluate model
# 1. Create the grid of feature values
exp_grid = np.linspace(
    players_untidy["experience_enc"].min() * 0.95,
    players_untidy["experience_enc"].max() * 1.05,
    50
)

age_grid = np.linspace(
    players_untidy["age"].min() * 0.95,
    players_untidy["age"].max() * 1.05,
    50
)

grid = np.array(np.meshgrid(exp_grid, age_grid)).reshape(2, -1).T
grid = pd.DataFrame(grid, columns=["experience_enc", "age"])


# 2. Predict at the grid points
grid_preds = knn_tune_grid.predict(grid) #todo

# 3. Bind predictions to the grid
prediction_table = grid.copy()
prediction_table["predicted"] = grid_preds


# 4. Plot original data
data_plot = alt.Chart(players_untidy).mark_point(
    opacity=0.6,
    filled=True,
    size=40
).encode(
    x=alt.X(
        "experience_enc",
        scale=alt.Scale(
            nice=False,
            domain=[
                players_untidy["experience_enc"].min() * 0.95,
                players_untidy["experience_enc"].max() * 1.05
            ]
        )
    ),
    y=alt.Y(
        "age",
        scale=alt.Scale(
            nice=False,
            domain=[
                players_untidy["age"].min() * 0.95,
                players_untidy["age"].max() * 1.05
            ]
        )
    ),
    color=alt.Color("subscribe:N", title="Actual")
)


# 5. Plot prediction background
prediction_plot = alt.Chart(prediction_table).mark_point(
    opacity=0.05,
    filled=True,
    size=300
).encode(
    x="experience_enc",
    y="age",
    color=alt.Color("predicted:N", title="Predicted")
)


# 6. Combine plots
data_plot + prediction_plot